# 02 Silver Layer
**EAS 587 Phase 3 | Drug Overdose CDC Dataset**

Apply cleaning and transformations. Write Silver Delta tables.




In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

DB_NAME = "eas587_phase3"

## Silver 1 — CDC Drug Overdose Deaths

In [0]:
bronze_cdc = spark.table(f"{DB_NAME}.bronze_cdc_overdose")
print(f"Bronze rows: {bronze_cdc.count():,}")
bronze_cdc.printSchema()

Bronze rows: 81,270
root
 |-- State: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Period: string (nullable = true)
 |-- Indicator: string (nullable = true)
 |-- Data Value: string (nullable = true)
 |-- Percent Complete: string (nullable = true)
 |-- Percent Pending Investigation: string (nullable = true)
 |-- State Name: string (nullable = true)
 |-- Footnote: string (nullable = true)
 |-- Footnote Symbol: string (nullable = true)
 |-- Predicted Value: string (nullable = true)



In [0]:
# 1: Rename columns to snake_case 


cdc = (
    bronze_cdc
    .withColumnRenamed("State",                         "state_code")
    .withColumnRenamed("State Name",                    "state_name")
    .withColumnRenamed("Indicator",                     "indicator")
    .withColumnRenamed("Data Value",                   "death_count_raw")
    .withColumnRenamed("Predicted Value",               "predicted_value_raw")
    .withColumnRenamed("Percent Complete",              "pct_complete_raw")
    .withColumnRenamed("Percent Pending Investigation", "pct_pending_raw")
)
cdc.show(5)

+----------+----+--------+---------------+---------------+---------------+----------------+---------------+----------+--------------------+---------------+-------------------+
|state_code|Year|   Month|         Period|      indicator|death_count_raw|pct_complete_raw|pct_pending_raw|state_name|            Footnote|Footnote Symbol|predicted_value_raw|
+----------+----+--------+---------------+---------------+---------------+----------------+---------------+----------+--------------------+---------------+-------------------+
|        AK|2015| January|12 month-ending|Cocaine (T40.5)|           NULL|             100|              0|    Alaska|Numbers may diffe...|             **|               NULL|
|        AK|2015|February|12 month-ending|Cocaine (T40.5)|           NULL|             100|              0|    Alaska|Numbers may diffe...|             **|               NULL|
|        AK|2015|   March|12 month-ending|Cocaine (T40.5)|           NULL|             100|              0|    Alaska|Nu

In [0]:
#  2: Strip commas and cast to numeric 

cdc = (
    cdc
    .withColumn("death_count",
        F.regexp_replace(F.col("death_count_raw"), ",", "").cast(DoubleType())
    )
    .withColumn("predicted_value",
        F.regexp_replace(F.col("predicted_value_raw"), ",", "").cast(DoubleType())
    )
    .withColumn("pct_complete", F.col("pct_complete_raw").cast(DoubleType()))
    .withColumn("pct_pending",  F.col("pct_pending_raw").cast(DoubleType()))
)


In [0]:
#  3: Parsing date 

from pyspark.sql.functions import col, concat_ws, to_date, lit

cdc_date_clean = cdc.withColumn(
    "Date",
    to_date(
        concat_ws(" ", col("Year"), col("Month"), lit("01")),
        "yyyy MMMM dd"
    )
)


cdc_date_clean.show(5, truncate=False)

+----------+----+--------+---------------+---------------+---------------+----------------+---------------+----------+------------------------------------------------------------------------------------------------------------------------+---------------+-------------------+----------+-----------+---------------+------------+-----------+
|state_code|Year|Month   |Period         |indicator      |death_count_raw|pct_complete_raw|pct_pending_raw|state_name|Footnote                                                                                                                |Footnote Symbol|predicted_value_raw|Date      |death_count|predicted_value|pct_complete|pct_pending|
+----------+----+--------+---------------+---------------+---------------+----------------+---------------+----------+------------------------------------------------------------------------------------------------------------------------+---------------+-------------------+----------+-----------+---------------+------

In [0]:
#  4: Quality filter 

cdc_filtered = cdc.filter(
    (F.col("pct_complete") == 100) &
    (F.col("pct_pending") < 0.3)
)
print(f"After quality filter: {cdc_filtered.count():,}  (was {cdc.count():,})")

After quality filter: 72,486  (was 81,270)


In [0]:
# 5: Drop nulls 
KEY_COLS = ["state_code", "indicator", "Date", "death_count", "predicted_value"]
cdc_no_nulls = cdc_filtered.dropna(subset=KEY_COLS)
print(f"After dropna : {cdc_no_nulls.count():,}")

After dropna : 47,365


In [0]:
#  6: Deduplicate 
cdc_deduped = cdc_no_nulls.dropDuplicates(["state_code", "indicator", "Date"])
print(f"After dedup  : {cdc_deduped.count():,}")

After dedup  : 47,365


In [0]:
#  7: Cleaning strings + adding drug_category label + select final columns
silver_cdc = (
    cdc_deduped
    .withColumn("state_code", F.upper(F.trim(F.col("state_code"))))
    .withColumn("state_name", F.initcap(F.trim(F.col("state_name"))))
    .withColumn("indicator",  F.trim(F.col("indicator")))
    # Derived: short drug category label used for ML encoding and joins
    .withColumn("drug_category",
        F.when(F.col("indicator").contains("Cocaine"),          F.lit("cocaine"))
         .when(F.col("indicator").contains("Heroin"),           F.lit("heroin"))
         .when(F.col("indicator").contains("Methadone"),        F.lit("methadone"))
         .when(F.col("indicator").contains("Synthetic"),        F.lit("synthetic_opioids"))
         .when(F.col("indicator").contains("Psychostimulants"), F.lit("psychostimulants"))
         .when(F.col("indicator").contains("Number of Drug"),   F.lit("total_overdose"))
         .otherwise(F.lit("other_opioids"))
    )
    .select(
        "state_code", "state_name",
        "indicator", "drug_category",
        "date","year",
        "death_count", "predicted_value",
        "pct_complete", "pct_pending"
    )
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window



w = Window.partitionBy("state_name", "indicator").orderBy("date")  # use your actual time column name

df_unrolled = (
    
    silver_cdc.withColumn(
        "actual_monthly_deaths",
        F.col("death_count") - F.lag("death_count", 12).over(w)
    )
)



silver_cdc = (
    df_unrolled
    .filter(F.col("actual_monthly_deaths").isNotNull())
    .filter(F.col("actual_monthly_deaths") >= 0)
)
print(f"\nFinal silver_cdc rows: {silver_cdc.count():,}")
silver_cdc.printSchema()
silver_cdc.show(5, truncate=False)


Final silver_cdc rows: 23,022
root
 |-- state_code: string (nullable = true)
 |-- state_name: string (nullable = true)
 |-- indicator: string (nullable = true)
 |-- drug_category: string (nullable = false)
 |-- date: date (nullable = false)
 |-- year: string (nullable = true)
 |-- death_count: double (nullable = true)
 |-- predicted_value: double (nullable = true)
 |-- pct_complete: double (nullable = true)
 |-- pct_pending: double (nullable = true)
 |-- actual_monthly_deaths: double (nullable = true)

+----------+----------+---------------+-------------+----------+----+-----------+---------------+------------+-------------+---------------------+
|state_code|state_name|indicator      |drug_category|date      |year|death_count|predicted_value|pct_complete|pct_pending  |actual_monthly_deaths|
+----------+----------+---------------+-------------+----------+----+-----------+---------------+------------+-------------+---------------------+
|AL        |Alabama   |Cocaine (T40.5)|cocaine    

In [0]:
# Null check
from pyspark.sql.functions import count, when, col
silver_cdc.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in silver_cdc.columns]
).show()

# Drug category distribution
silver_cdc.groupBy("drug_category").count().orderBy("drug_category").show()

# Date range
silver_cdc.agg(F.min("date").alias("min"), F.max("date").alias("max")).show()

+----------+----------+---------+-------------+----+----+-----------+---------------+------------+-----------+---------------------+
|state_code|state_name|indicator|drug_category|date|year|death_count|predicted_value|pct_complete|pct_pending|actual_monthly_deaths|
+----------+----------+---------+-------------+----+----+-----------+---------------+------------+-----------+---------------------+
|         0|         0|        0|            0|   0|   0|          0|              0|           0|          0|                    0|
+----------+----------+---------+-------------+----+----+-----------+---------------+------------+-----------+---------------------+

+-----------------+-----+
|    drug_category|count|
+-----------------+-----+
|          cocaine| 2735|
|           heroin| 1163|
|        methadone| 1810|
|    other_opioids| 8148|
| psychostimulants| 2865|
|synthetic_opioids| 2784|
|   total_overdose| 3517|
+-----------------+-----+

+----------+----------+
|       min|       max|

In [0]:
(
    silver_cdc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB_NAME}.silver_cdc_overdose")
)
print("✅ silver_cdc_overdose written")

✅ silver_cdc_overdose written


---
## Silver 2 — KFF Opioid Use Disorder by State

Source: SAMHSA NSDUH 2022–2023 via Kaiser Family Foundation  
3 columns, 51 rows (50 states + DC). Values are decimal rates (e.g. 0.027 = 2.7%).



In [0]:
bronze_kff = spark.table(f"{DB_NAME}.bronze_kff_opioid_disorder")
print(f"Bronze KFF rows: {bronze_kff.count()}")
bronze_kff.show(truncate=False)

Bronze KFF rows: 62
+--------------------+-------------------------------------+----------------------------------+
|Location            |Adolescents_12_17_Opioid_Use_Disorder|Adults_18_Plus_Opioid_Use_Disorder|
+--------------------+-------------------------------------+----------------------------------+
|Alabama             |0.01                                 |0.027                             |
|Alaska              |0.008                                |0.022                             |
|Arizona             |0.013                                |0.024                             |
|Arkansas            |0.014                                |0.023                             |
|California          |0.011                                |0.019                             |
|Colorado            |0.01                                 |0.021                             |
|Connecticut         |0.01                                 |0.021                             |
|Delaware           

In [0]:


# Rename to short snake_case
kff = (
    bronze_kff
    .withColumnRenamed("Location",                                 "state_name_raw")
    .withColumnRenamed("Adolescents_12_17_Opioid_Use_Disorder",    "adolescent_oud_rate_raw")
    .withColumnRenamed("Adults_18_Plus_Opioid_Use_Disorder",       "adult_oud_rate_raw")
)

 #1: Cast to numeric (mirrors pd.to_numeric with errors='coerce') 
kff = (
    kff
    .withColumn("adolescent_oud_rate", F.col("adolescent_oud_rate_raw").cast(DoubleType()))
    .withColumn("adult_oud_rate",      F.col("adult_oud_rate_raw").cast(DoubleType()))
)

#  2: Drop nulls 
kff = kff.dropna(subset=["state_name_raw", "adolescent_oud_rate", "adult_oud_rate"])

# 3: Drop duplicates 
kff = kff.dropDuplicates(["state_name_raw"])

#  4: Standardize state_name 
kff = kff.withColumn("state_name", F.initcap(F.trim(F.col("state_name_raw"))))

# 5: Convert decimal rates to percentage + add combined rate 

kff = (
    kff
    .withColumn("adolescent_oud_pct",  F.round(F.col("adolescent_oud_rate") * 100, 3))
    .withColumn("adult_oud_pct",       F.round(F.col("adult_oud_rate")      * 100, 3))
    # Simple average: (adolescent + adult) / 2  — overall state OUD burden proxy
    .withColumn("combined_oud_pct",
        F.round((F.col("adolescent_oud_pct") + F.col("adult_oud_pct")) / 2, 3)
    )
    # OUD risk tier: high / medium / low based on adult rate (primary risk indicator)
    .withColumn("oud_risk_tier",
        F.when(F.col("adult_oud_pct") >= 3.0, F.lit("high"))
         .when(F.col("adult_oud_pct") >= 2.2, F.lit("medium"))
         .otherwise(F.lit("low"))
    )
    .select(
        "state_name",
        "adolescent_oud_pct", "adult_oud_pct", "combined_oud_pct",
        "oud_risk_tier"
    )
)

print(f"Silver KFF rows: {kff.count()}")
kff.orderBy(F.desc("adult_oud_pct")).show(truncate=False)

Silver KFF rows: 51
+--------------+------------------+-------------+----------------+-------------+
|state_name    |adolescent_oud_pct|adult_oud_pct|combined_oud_pct|oud_risk_tier|
+--------------+------------------+-------------+----------------+-------------+
|Louisiana     |1.2               |4.0          |2.6             |high         |
|West Virginia |0.9               |3.7          |2.3             |high         |
|Mississippi   |1.0               |3.6          |2.3             |high         |
|Kentucky      |1.4               |3.3          |2.35            |high         |
|New Mexico    |1.0               |3.0          |2.0             |high         |
|Oklahoma      |1.4               |2.8          |2.1             |medium       |
|Alabama       |1.0               |2.7          |1.85            |medium       |
|Nevada        |1.2               |2.6          |1.9             |medium       |
|South Carolina|1.3               |2.5          |1.9             |medium       |
|North C

In [0]:
# OUD risk tier distribution
kff.groupBy("oud_risk_tier").count().show()

# Top 10 states by adult OUD rate
kff.orderBy(F.desc("adult_oud_pct")).show(10, truncate=False)

+-------------+-----+
|oud_risk_tier|count|
+-------------+-----+
|       medium|   22|
|          low|   24|
|         high|    5|
+-------------+-----+

+--------------+------------------+-------------+----------------+-------------+
|state_name    |adolescent_oud_pct|adult_oud_pct|combined_oud_pct|oud_risk_tier|
+--------------+------------------+-------------+----------------+-------------+
|Louisiana     |1.2               |4.0          |2.6             |high         |
|West Virginia |0.9               |3.7          |2.3             |high         |
|Mississippi   |1.0               |3.6          |2.3             |high         |
|Kentucky      |1.4               |3.3          |2.35            |high         |
|New Mexico    |1.0               |3.0          |2.0             |high         |
|Oklahoma      |1.4               |2.8          |2.1             |medium       |
|Alabama       |1.0               |2.7          |1.85            |medium       |
|Nevada        |1.2               |

In [0]:
(
    kff.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB_NAME}.silver_kff_opioid_disorder")
)
print("✅ silver_kff_opioid_disorder written")

✅ silver_kff_opioid_disorder written
